# 05 — Train RF-DETR

Trains RF-DETR-Medium on the BDD100K clear-weather training split.
RF-DETR uses a **DINOv2 ViT-B/14 backbone** and the `rfdetr` Roboflow package.
This notebook uses the **COCO-format** dataset (see `01_setup_and_data.ipynb`).

**Note:** RF-DETR does NOT use the Ultralytics API — it has its own training interface.

**Prerequisites:** run `01_setup_and_data.ipynb` first.

In [1]:
!pip install -q "rfdetr[train,loggers]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from pathlib import Path
import torch
from dotenv import load_dotenv
from src.train_utils import setup_logging

DRIVE_ROOT    = Path('/content/drive/MyDrive/FON/master_rad')
COCO_DATA_DIR = Path('/content/data_prepared/coco')
CHECKPOINTS   = DRIVE_ROOT / 'checkpoints'

load_dotenv(DRIVE_ROOT / '.env')
logger = setup_logging()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('COCO data dir exists:', COCO_DATA_DIR.exists())

Device: cuda
COCO data dir exists: True


In [3]:
# RF-DETR expects _annotations.coco.json and valid/ (not annotations.json / val/)
# This cell migrates already-prepared data in-place.
coco_clear = COCO_DATA_DIR / 'clear_day'
for src_name, dst_name in [('train', 'train'), ('val', 'valid')]:
    src_dir = coco_clear / src_name
    dst_dir = coco_clear / dst_name
    if src_name != dst_name and src_dir.exists() and not dst_dir.exists():
        src_dir.rename(dst_dir)
        print(f'Renamed {src_name}/ → {dst_name}/')
    old_ann = dst_dir / 'annotations.json'
    new_ann = dst_dir / '_annotations.coco.json'
    if old_ann.exists() and not new_ann.exists():
        old_ann.rename(new_ann)
        print(f'Renamed annotations.json → _annotations.coco.json in {dst_name}/')
print('COCO data ready:', list(coco_clear.iterdir()))

Renamed annotations.json → _annotations.coco.json in train/
Renamed val/ → valid/
Renamed annotations.json → _annotations.coco.json in valid/
COCO data ready: [PosixPath('/content/data_prepared/coco/clear_day/train'), PosixPath('/content/data_prepared/coco/clear_day/valid')]


In [4]:
from rfdetr import RFDETRMedium

CHECKPOINT_DIR = CHECKPOINTS / 'rfdetr'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Resume from latest checkpoint if one exists
checkpoints = sorted(CHECKPOINT_DIR.glob('*.pth'))
resume_path = str(checkpoints[-1]) if checkpoints else None
if resume_path:
    print(f'Resuming from: {resume_path}')
else:
    print('Starting from scratch.')

model = RFDETRMedium()

results = model.train(
    dataset_dir=str(COCO_DATA_DIR / 'clear_day'),
    epochs=50,
    batch_size=16,
    grad_accum_steps=1,
    lr=1e-4,
    lr_encoder=1.5e-4,
    weight_decay=1e-4,
    resolution=576,
    output_dir=str(CHECKPOINT_DIR),
    checkpoint_interval=1,
    eval_interval=2,
    fp16_eval=True,
    use_ema=True,
    early_stopping=True,
    early_stopping_patience=10,
    resume=resume_path,
)
print('Training complete.')

Resuming from: /content/drive/MyDrive/FON/master_rad/checkpoints/rfdetr/checkpoint_best_regular.pth
[2026-05-05 17:12:43] [INFO] rf-detr - Downloading pretrained weights for rf-detr-medium.pth


rf-detr-medium.pth:   0%|          | 0.00/386M [00:00<?, ?iB/s]

[2026-05-05 17:12:54] [INFO] rf-detr - MD5 validation successful for rf-detr-medium.pth


[2026-05-05 17:12:54] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-05-05 17:12:54] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-05-05 17:12:56] [INFO] rf-detr - File rf-detr-medium.pth already exists with correct MD5 hash.


...

21:10:22 | INFO     | Evaluate annotation type *bbox*
21:10:30 | INFO     | COCOeval_opt.evaluate() finished...
21:10:30 | INFO     | DONE (t=8.16s).
21:10:30 | INFO     | Accumulating evaluation results...
21:10:30 | INFO     | COCOeval_opt.accumulate() finished...
21:10:30 | INFO     | DONE (t=0.00s).
21:10:30 | INFO     |  Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=500 ] = 0.312
21:10:30 | INFO     |  Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=500 ] = 0.547
21:10:30 | INFO     |  Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=500 ] = 0.300
21:10:30 | INFO     |  Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=500 ] = 0.123
21:10:30 | INFO     |  Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=500 ] = 0.365
21:10:30 | INFO     |  Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=500 ] = 0.710
21:10:30 | INFO     |  Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | max

22:11:14 | INFO     | Evaluate annotation type *bbox*
22:11:22 | INFO     | COCOeval_opt.evaluate() finished...
22:11:22 | INFO     | DONE (t=8.13s).
22:11:22 | INFO     | Accumulating evaluation results...
22:11:22 | INFO     | COCOeval_opt.accumulate() finished...
22:11:22 | INFO     | DONE (t=0.00s).
22:11:22 | INFO     |  Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=500 ] = 0.324
22:11:22 | INFO     |  Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=500 ] = 0.570
22:11:22 | INFO     |  Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=500 ] = 0.316
22:11:22 | INFO     |  Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=500 ] = 0.126
22:11:22 | INFO     |  Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=500 ] = 0.356
22:11:22 | INFO     |  Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=500 ] = 0.698
22:11:22 | INFO     |  Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | max

23:13:57 | INFO     | Evaluate annotation type *bbox*
23:14:06 | INFO     | COCOeval_opt.evaluate() finished...
23:14:06 | INFO     | DONE (t=8.16s).
23:14:06 | INFO     | Accumulating evaluation results...
23:14:06 | INFO     | COCOeval_opt.accumulate() finished...
23:14:06 | INFO     | DONE (t=0.00s).
23:14:06 | INFO     |  Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=500 ] = 0.322
23:14:06 | INFO     |  Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=500 ] = 0.559
23:14:06 | INFO     |  Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=500 ] = 0.314
23:14:06 | INFO     |  Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=500 ] = 0.116
23:14:06 | INFO     |  Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=500 ] = 0.353
23:14:06 | INFO     |  Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=500 ] = 0.708
23:14:06 | INFO     |  Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | max

23:16:22 | INFO     | Monitored metric __rfdetr_effective_map__ did not improve in the last 10 records. Best score: 0.342. Signaling Trainer to stop.


[2026-05-05 23:16:28] [INFO] rf-detr - Best total checkpoint saved from EMA (regular=0.3339, ema=0.3422)
Training complete.


In [5]:
import json
import pandas as pd

metrics_csv = CHECKPOINT_DIR / 'metrics.csv'
best_map50 = best_map50_95 = None
if metrics_csv.exists():
    df = pd.read_csv(metrics_csv)
    best_map50    = float(df['val/ema_mAP_50'].max())
    best_map50_95 = float(df['val/ema_mAP_50_95'].max())

summary = {
    'model': 'rfdetr_medium',
    'best_map50': best_map50,
    'best_map50_95': best_map50_95,
    'epochs': 50,
    'resolution': 576,
    'batch_size': 16,
    'lr': 1e-4,
    'lr_encoder': 1.5e-4,
    'weight_decay': 1e-4,
}
out = CHECKPOINT_DIR / 'training_summary.json'
out.write_text(json.dumps(summary, indent=2))
print('Summary saved to', out)
print(f'Best mAP@50: {best_map50}')
print(f'Best mAP@50:95: {best_map50_95}')

Summary saved to /content/drive/MyDrive/FON/master_rad/checkpoints/rfdetr/training_summary.json
Best mAP@50: 0.5859192609786987
Best mAP@50:95: 0.3421684205532074
